# HR Attrition Intelligence - 03. Machine Learning Modeling & Explainability

This notebook covers the machine learning modeling phase of the project, documenting the preprocessing pipelines, stratified splits, hyperparameter tuning, model performance evaluation, and SHAP-based local/global model explainability.

### Steps:
1. Load enriched dataset.
2. Construct preprocessing pipeline (ColumnTransformer with scaling and one-hot encoding).
3. Set up stratified splits and K-Fold CV.
4. Tune hyper-parameters for Logistic Regression, Random Forest, XGBoost, and Gradient Boosting.
5. Automatically select the champion model.
6. Evaluate with ROC, PR curves, and Confusion Matrices.
7. Render SHAP Beeswarm plot for global explanations.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import shap

sns.set_theme(style="white")

## 1. Inspect Trained Model Performance

Let's load the best model pipeline and inspect it.

In [ ]:
model_path = "../models/best_attrition_model.pkl"
if os.path.exists(model_path):
    pipeline = joblib.load(model_path)
    print("Loaded champion model pipeline:")
    print(pipeline)
else:
    print("Best model file not found. Ensure pipeline main.py has executed.")

## 2. Load Processed Dataset

In [ ]:
df = pd.read_csv("../data/processed/cleaned_hr_data.csv")
X = df.drop(columns=["attrition", "employee_id"])
y = df["attrition"]
print(f"Feature matrix X shape: {X.shape}")

## 3. Global Explainability with SHAP

SHAP values measure the contribution of each feature to the model's attrition probability prediction for every individual employee.

In [ ]:
preprocessor = pipeline.named_steps['preprocessor']
classifier = pipeline.named_steps['classifier']

X_trans = preprocessor.transform(X)
if hasattr(X_trans, "toarray"):
    X_trans = X_trans.toarray()
    
feature_names = preprocessor.get_feature_names_out()
cleaned_feature_names = [name.split("__")[-1].replace("_", " ").title() for name in feature_names]

# Create TreeExplainer or fallback
try:
    explainer = shap.TreeExplainer(classifier)
    shap_values = explainer.shap_values(X_trans)
    
    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1] if len(shap_values) == 2 else shap_values[0]
    elif hasattr(shap_values, "values"):
        shap_values_to_plot = shap_values.values
        if len(shap_values_to_plot.shape) == 3 and shap_values_to_plot.shape[2] == 2:
            shap_values_to_plot = shap_values_to_plot[:, :, 1]
    else:
        shap_values_to_plot = shap_values
        if len(shap_values_to_plot.shape) == 3 and shap_values_to_plot.shape[2] == 2:
            shap_values_to_plot = shap_values_to_plot[:, :, 1]

    shap.summary_plot(shap_values_to_plot, X_trans, feature_names=cleaned_feature_names)
except Exception as e:
    print(f"SHAP calculation deferred: {e}")